# Projet 3 — Backtesting d'une stratégie systématique
## Notebook 08a — Données de l'univers S&P 500 élargi (screening de paires)

**Rôle.** On élargit l'univers de recherche de paires à ~200 grandes capitalisations du S&P 500, réparties dans 21 secteurs profonds (banques, assurances, utilities, REITs, semi-conducteurs, pharma, rails…). Plus l'univers est riche, plus on a de chances de trouver des paires réellement cointégrées et tradables. Le screener (notebook 08b) reprendra exactement le pipeline du 06b sur cet univers.

**Toujours hors univers momentum** (aucun des 8 titres AAPL, MSFT, NVDA, AMZN, JPM, XOM, JNJ, PG), pour garder les stratégies indépendantes.

**Point technique clé — la troncature.** Avec ~200 titres, certains ont une introduction en bourse tardive. Si on alignait toutes les séries sur leurs dates communes, la plus récente imposerait le début à toutes (ex. une IPO de 2022 ramènerait tout le monde à 2022). On **retire donc automatiquement** les titres démarrant après une date seuil (2012 par défaut) *avant* d'aligner, ce qui préserve un long historique commun. On affiche la liste des titres retirés.

> À exécuter sur ta machine (yfinance). Archivage `.parquet` comme d'habitude.


In [1]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
try:
    import yfinance as yf
except ImportError:
    raise ImportError("Installe yfinance : pip install yfinance")
plt.rcParams["figure.figsize"]=(11,4.5)
DATA_DIR="../data"; os.makedirs(DATA_DIR, exist_ok=True)

SECTORS = {
 "Banks": ["BAC","WFC","C","GS","MS","USB","PNC","TFC","COF","BK","STT","FITB","HBAN","RF","CFG","KEY","MTB","NTRS","ALLY"],
 "Insurance": ["BRK-B","PGR","TRV","ALL","MET","PRU","AIG","CB","AFL","HIG","PFG","AJG","MMC","AON","CINF","L"],
 "CapitalMarkets": ["SPGI","MCO","ICE","CME","MSCI","NDAQ","SCHW","BLK","TROW","AMP"],
 "Oil_Gas": ["CVX","COP","SLB","EOG","MPC","PSX","VLO","OXY","HES","DVN","FANG","HAL","BKR","WMB","KMI","OKE","TRGP"],
 "Utilities": ["NEE","DUK","SO","D","AEP","EXC","SRE","XEL","ED","WEC","ES","PEG","AEE","DTE","PPL","CMS","CNP","ATO"],
 "REITs": ["AMT","PLD","CCI","EQIX","PSA","O","SPG","WELL","DLR","VICI","AVB","EQR","SBAC","ARE","VTR","ESS","MAA"],
 "Semis": ["AMD","INTC","QCOM","TXN","MU","ADI","AMAT","LRCX","KLAC","MCHP","NXPI","ON","MPWR"],
 "Software": ["ORCL","CRM","ADBE","NOW","INTU","IBM","ADP","FIS","FI","CTSH","ACN"],
 "Pharma": ["LLY","ABBV","MRK","PFE","BMY","AMGN","GILD","VRTX","REGN","BIIB","ZTS"],
 "HealthEquip": ["ABT","TMO","DHR","MDT","SYK","BSX","BDX","ISRG","EW","ZBH","BAX"],
 "Payments_Cards": ["V","MA","AXP","PYPL","GPN"],
 "Telecom": ["VZ","T","TMUS"],
 "Retail_Big": ["HD","LOW","TGT","WMT","COST","DG","DLTR","BBY"],
 "Consumer_Staples": ["KO","PEP","MDLZ","CL","KMB","GIS","KHC","HSY","STZ","K","SYY","ADM"],
 "Food_Restaurant": ["MCD","SBUX","CMG","YUM","DRI"],
 "Industrials_Machinery": ["CAT","DE","HON","GE","MMM","EMR","ITW","ETN","PH","ROK","DOV","CMI"],
 "Aerospace_Defense": ["BA","LMT","RTX","NOC","GD","LHX","TDG","HWM"],
 "Rails_Transport": ["UNP","CSX","NSC","FDX","UPS","ODFL"],
 "Autos": ["TSLA","F","GM","APTV","BWA"],
 "Chemicals": ["LIN","APD","SHW","ECL","DD","DOW","PPG","NEM","FCX"],
 "Media_Comm": ["GOOGL","GOOG","META","NFLX","DIS","CMCSA","CHTR","WBD","TTWO","EA"],
}
TICKERS = sorted({t for v in SECTORS.values() for t in v})
MOMENTUM = {"AAPL","MSFT","NVDA","AMZN","JPM","XOM","JNJ","PG"}
assert not (MOMENTUM & set(TICKERS)), "Chevauchement interdit avec l'univers momentum"
print(f"{len(TICKERS)} titres, {len(SECTORS)} secteurs, aucun chevauchement momentum.")

START="2010-01-01"; END=None
CUTOFF="2012-01-01"   # on retire les titres démarrant après cette date

226 titres, 21 secteurs, aucun chevauchement momentum.


In [2]:
RAW_PATH = os.path.join(DATA_DIR, "sp500_prices_raw.parquet")

def download_prices(tickers, start, end, path, force_download=False):
    if os.path.exists(path) and not force_download:
        print(f"Chargement du cache local : {path}")
        return pd.read_parquet(path)
    print(f"Téléchargement de {len(tickers)} titres via yfinance (peut prendre 1-2 min)...")
    raw = yf.download(tickers, start=start, end=end, auto_adjust=False,
                      group_by="column", progress=False, threads=True)
    if raw.empty:
        raise RuntimeError("Téléchargement vide.")
    raw.to_parquet(path)
    print(f"Archivé : {path}  (shape={raw.shape})")
    return raw

raw = download_prices(TICKERS, START, END, RAW_PATH)

Téléchargement de 226 titres via yfinance (peut prendre 1-2 min)...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FI"}}}
$FI: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HES"}}}
$K: possibly delisted; no timezone found
$HES: possibly delisted; no timezone found
$MMC: possibly delisted; no timezone found
$BK: possibly delisted; no timezone found

5 Failed downloads:
['FI', 'K', 'HES', 'MMC', 'BK']: possibly delisted; no timezone found


Archivé : ../data\sp500_prices_raw.parquet  (shape=(4184, 1356))


In [3]:
def get_field(raw, field):
    if isinstance(raw.columns, pd.MultiIndex): return raw[field].copy()
    return raw[[field]].copy()

adj_raw = get_field(raw, "Adj Close")

# Titres réellement récupérés (certains tickers peuvent échouer)
missing = [t for t in TICKERS if t not in adj_raw.columns or adj_raw[t].notna().sum() == 0]
print("Titres non récupérés (ignorés) :", missing if missing else "aucun")
adj_raw = adj_raw[[c for c in adj_raw.columns if c not in missing]]

starts = adj_raw.apply(lambda s: s.first_valid_index())
late = starts[starts > CUTOFF].sort_values()
print(f"\nTitres démarrant après {CUTOFF} (retirés pour préserver l'historique) :")
print(late.to_string() if len(late) else "  aucun")
adj_raw = adj_raw.drop(columns=late.index.tolist())
print(f"\nTitres conservés : {adj_raw.shape[1]}")

Titres non récupérés (ignorés) : ['BK', 'FI', 'HES', 'K', 'MMC']

Titres démarrant après 2012-01-01 (retirés pour préserver l'historique) :
Ticker
PSX    2012-04-12
META   2012-05-18
NOW    2012-06-29
FANG   2012-10-12
ABBV   2013-01-02
ZTS    2013-02-01
ALLY   2014-01-28
CFG    2014-09-24
KHC    2015-07-06
PYPL   2015-07-06
HWM    2016-11-01
VICI   2018-01-02
DOW    2019-03-20
EA     2026-07-17

Titres conservés : 207


In [4]:
def clean_prices(df, ffill_limit=5):
    df = df[~df.index.duplicated(keep="first")].sort_index()
    df = df.ffill(limit=ffill_limit)
    df = df.dropna(how="any")
    return df

adj_clean = clean_prices(adj_raw)
print("Shape après nettoyage/alignement :", adj_clean.shape)
print(f"Période effective : {adj_clean.index.min().date()} -> {adj_clean.index.max().date()}")
print("Valeurs manquantes :", int(adj_clean.isna().sum().sum()))

Shape après nettoyage/alignement : (3709, 207)
Période effective : 2011-11-17 -> 2026-08-21
Valeurs manquantes : 0


## Sauvegarde
On archive les cours propres et la table secteur (restreinte aux titres conservés). Le screener 08b chargera ces deux fichiers.


In [5]:
adj_clean.to_csv(os.path.join(DATA_DIR, "sp500_adj_close.csv"))
sector_map = {t: sec for sec, ts in SECTORS.items() for t in ts if t in adj_clean.columns}
pd.Series(sector_map, name="secteur").rename_axis("ticker").to_csv(
    os.path.join(DATA_DIR, "sp500_sectors.csv"))

# nb de paires intra-secteur qui seront testées
from itertools import combinations
kept = pd.Series(sector_map)
npairs = sum(len(list(combinations(kept[kept==s].index, 2))) for s in kept.unique())
print(f"Fichiers écrits. Titres conservés : {adj_clean.shape[1]}")
print(f"Paires intra-secteur à tester au screening : {npairs}")
for f in ["sp500_prices_raw.parquet","sp500_adj_close.csv","sp500_sectors.csv"]:
    p=os.path.join(DATA_DIR,f)
    if os.path.exists(p): print(f"  - {f}  ({os.path.getsize(p)/1024:.0f} Ko)")
print("\nPrêt pour le notebook 08b — screener sur l'univers S&P 500 élargi.")

Fichiers écrits. Titres conservés : 207
Paires intra-secteur à tester au screening : 1099
  - sp500_prices_raw.parquet  (34297 Ko)
  - sp500_adj_close.csv  (13700 Ko)
  - sp500_sectors.csv  (3 Ko)

Prêt pour le notebook 08b — screener sur l'univers S&P 500 élargi.
